# Fair Value Gap (FVG) Strategy — Backtest Research

**Instrument:** QQQ (1-min bars)
**Signal:** 3-bar FVG pattern after the opening range, with retracement + engulfing entry
**Directions:** Bullish (long) and Bearish (short) tested independently
**Exit:** 1% stop loss, 1.5R profit target, or EOD

**Conclusion:** Modest edge (~58% WR bullish, ~20% total return). Not compelling
enough to pursue further, but documented here for reference.

Uses `shared/` for data fetching, fees, metrics, and plotting.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

from shared.data import fetch_historical_data
from shared.fees import calculate_fees
from shared.metrics import evaluate_strategy, print_metrics, compare_strategies
from shared.plotting import plot_equity_curve, plot_trade_returns, plot_yearly_returns, plot_equity_comparison
from shared.results import save_trades

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

SYMBOL     = "QQQ"
START_DATE = "2016-01-01"
END_DATE   = "2026-04-01"

# Backtest parameters
STARTING_CAPITAL = 100_000
RISK_PERCENT     = 0.01
LEVERAGE         = 1
MAX_CAP_PERCENT  = 1
INCLUDE_FEES     = True

# FVG pattern parameters
OPENING_RANGE_MINUTES = 5
EXPANSION_RATIO       = 1.75
CONSOLIDATION_RATIO   = 1.0
STOP_LOSS_PCT         = 0.01   # 1%
REWARD_RISK_RATIO     = 1.5    # 1.5R target

## 2. Data Fetching

In [ ]:
data_dict = fetch_historical_data(
    [SYMBOL], TimeFrame(1, TimeFrameUnit.Minute), START_DATE, END_DATE)

qqq_data = data_dict[SYMBOL].reset_index()

# Ensure timezone
if qqq_data["timestamp"].dt.tz is None:
    qqq_data["timestamp"] = qqq_data["timestamp"].dt.tz_localize("UTC")
qqq_data["timestamp"] = qqq_data["timestamp"].dt.tz_convert("America/New_York")

# Filter to regular hours
qqq_data = qqq_data[
    (qqq_data["timestamp"].dt.time >= pd.Timestamp("09:30").time()) &
    (qqq_data["timestamp"].dt.time <= pd.Timestamp("16:00").time())
].copy()

print(f"Regular-session 1-min bars: {len(qqq_data):,}")
print(f"Trading days: {qqq_data['timestamp'].dt.date.nunique():,}")

## 3. Strategy Rules

### Opening Range (9:30 – 9:35)
Aggregate first 5 minutes. **H0** = high, **L0** = low.

### FVG Formation — Three-Bar Pattern
**Bullish:** Bar 1 opens below H0 → Bar 2 is 1.75× expansion, bullish, gaps up → Bar 3 consolidates, gap maintained, closes above H0.
**Bearish:** Mirror of bullish.

### Retracement & Entry
Wait for counter-trend bar entering the FVG zone, then multi-bar engulfing entry.

### Exit
1% stop, 1.5R target, or EOD.

## 4. FVG Detection & Signal Generation

In [ ]:
def compute_opening_range(day_data):
    """Aggregate 9:30-9:35 bars into opening range OHLC."""
    or_data = day_data[
        (day_data["timestamp"].dt.time >= pd.Timestamp("09:30").time()) &
        (day_data["timestamp"].dt.time <= pd.Timestamp("09:35").time())
    ]
    if or_data.empty:
        return None
    return {
        "open": or_data.iloc[0]["open"],
        "high": or_data["high"].max(),
        "low": or_data["low"].min(),
        "close": or_data.iloc[-1]["close"],
    }


def detect_fvg_signals(day_data, direction, H0, L0):
    """
    Detect FVG pattern and entry signal for a single day.

    Parameters
    ----------
    day_data  : pd.DataFrame — 1-min bars for the day.
    direction : str          — 'long' or 'short'.
    H0, L0    : float        — opening range high/low.

    Returns
    -------
    dict with trade details, or None if no signal.
    """
    remaining = day_data[
        day_data["timestamp"].dt.time > pd.Timestamp("09:35").time()
    ].reset_index(drop=True)

    is_long = direction == "long"

    # Phase 1: Find 3-bar FVG pattern
    expansion = False
    H1 = L1 = O1 = C1 = None
    H2 = L2 = O2 = C2 = None
    bar1_time = None

    for i, row in remaining.iterrows():
        H_i, L_i, O_i, C_i = row["high"], row["low"], row["open"], row["close"]

        if not expansion:
            if H1 is None:
                if is_long and O_i >= H0:
                    continue
                if not is_long and O_i <= L0:
                    continue
                H1, L1, O1, C1 = H_i, L_i, O_i, C_i
                bar1_time = row["timestamp"]
                bar1_idx = i
            else:
                if (H1 - L1) == 0:
                    H1, L1, O1, C1 = H_i, L_i, O_i, C_i
                    bar1_time = row["timestamp"]
                    bar1_idx = i
                    if is_long and O1 >= H0:
                        H1 = L1 = O1 = C1 = None
                    if not is_long and O1 <= L0:
                        H1 = L1 = O1 = C1 = None
                    continue

                size_ratio = (H_i - L_i) / (H1 - L1)
                if is_long:
                    exp_ok = size_ratio > EXPANSION_RATIO and C_i > O_i
                    gap_ok = O_i > C1
                else:
                    exp_ok = size_ratio > EXPANSION_RATIO and C_i < O_i
                    gap_ok = O_i < C1

                if exp_ok and gap_ok:
                    expansion = True
                    H2, L2, O2, C2 = H_i, L_i, O_i, C_i
                    bar2_idx = i
                else:
                    H1, L1, O1, C1 = H_i, L_i, O_i, C_i
                    bar1_time = row["timestamp"]
                    bar1_idx = i
                    if is_long and O1 >= H0:
                        H1 = L1 = O1 = C1 = None
                    if not is_long and O1 <= L0:
                        H1 = L1 = O1 = C1 = None
        else:
            H3, L3, O3, C3 = H_i, L_i, O_i, C_i
            if (H2 - L2) == 0:
                expansion = False
                H1, L1, O1, C1 = H3, L3, O3, C3
                bar1_time = row["timestamp"]
                bar1_idx = i
                H2 = L2 = O2 = C2 = None
                if is_long and O1 >= H0:
                    H1 = L1 = O1 = C1 = None
                if not is_long and O1 <= L0:
                    H1 = L1 = O1 = C1 = None
                continue

            consol_ok = (H3 - L3) / (H2 - L2) < CONSOLIDATION_RATIO
            if is_long:
                gap_kept = L3 > H1
                confirm = C3 > H0
                fvg_range = [H1, L3]
            else:
                gap_kept = H3 < L1
                confirm = C3 < L0
                fvg_range = [H3, L1]

            if consol_ok and gap_kept and confirm:
                fvg_complete_time = row["timestamp"]
                break
            else:
                expansion = False
                H1, L1, O1, C1 = H3, L3, O3, C3
                bar1_time = row["timestamp"]
                bar1_idx = i
                H2 = L2 = O2 = C2 = None
                if is_long and O1 >= H0:
                    H1 = L1 = O1 = C1 = None
                if not is_long and O1 <= L0:
                    H1 = L1 = O1 = C1 = None
    else:
        return None

    # Phase 2: Retracement + engulfing
    retracement = False
    retracement_bar = None
    engulfing_bars = []

    for j in range(i + 1, len(remaining)):
        bar = remaining.iloc[j]
        if not retracement:
            if is_long:
                retrace_ok = (bar["close"] < bar["open"]) and (bar["low"] < fvg_range[1])
            else:
                is_bull = bar["close"] > bar["open"]
                enters = bar["high"] > fvg_range[0] and bar["high"] < fvg_range[1]
                breaks = bar["close"] > fvg_range[1]
                if breaks:
                    return None
                retrace_ok = is_bull and enters

            if retrace_ok:
                retracement = True
                retracement_bar = {"o": bar["open"], "h": bar["high"],
                                   "l": bar["low"], "c": bar["close"],
                                   "time": bar["timestamp"]}
        else:
            if is_long:
                is_cont = bar["close"] > bar["open"]
            else:
                is_cont = bar["close"] < bar["open"]

            if is_cont:
                engulfing_bars.append({"open": bar["open"], "close": bar["close"],
                                       "high": bar["high"], "low": bar["low"],
                                       "time": bar["timestamp"]})
                if is_long:
                    min_o = min(b["open"] for b in engulfing_bars)
                    max_c = max(b["close"] for b in engulfing_bars)
                    engulfs = (min_o <= retracement_bar["c"]) and (max_c >= retracement_bar["o"])
                else:
                    max_o = max(b["open"] for b in engulfing_bars)
                    min_c = min(b["close"] for b in engulfing_bars)
                    engulfs = (max_o >= retracement_bar["c"]) and (min_c <= retracement_bar["o"])
            else:
                engulfing_bars = []
                engulfs = False

            if engulfs and j + 1 < len(remaining):
                entry_bar = remaining.iloc[j + 1]
                entry_price = entry_bar["open"]

                if is_long:
                    stop_loss = entry_price * (1 - STOP_LOSS_PCT)
                    risk = entry_price - stop_loss
                    profit_target = entry_price + REWARD_RISK_RATIO * risk
                else:
                    stop_loss = entry_price * (1 + STOP_LOSS_PCT)
                    risk = stop_loss - entry_price
                    profit_target = entry_price - REWARD_RISK_RATIO * risk

                return {
                    "direction": direction,
                    "fvg_formation_time": bar1_time,
                    "fvg_complete_time": fvg_complete_time,
                    "retracement_time": retracement_bar["time"],
                    "signal_time": bar["timestamp"],
                    "entry_time": entry_bar["timestamp"],
                    "entry_price": entry_price,
                    "stop_loss": stop_loss,
                    "profit_target": profit_target,
                    "risk": risk,
                    "fvg_range": fvg_range,
                    "H0": H0, "L0": L0, "H1": H1, "L1": L1,
                }
    return None

## 5. Scan All Trading Days

In [ ]:
dates = qqq_data["timestamp"].dt.date.unique()

all_signals = []
for date in dates:
    day = qqq_data[qqq_data["timestamp"].dt.date == date].reset_index(drop=True)
    opening = compute_opening_range(day)
    if opening is None:
        continue
    H0, L0 = opening["high"], opening["low"]
    for direction in ["long", "short"]:
        signal = detect_fvg_signals(day, direction, H0, L0)
        if signal is not None:
            signal["date"] = date
            all_signals.append(signal)

signals_df = pd.DataFrame(all_signals)
print(f"Total FVG signals: {len(signals_df)}")
if not signals_df.empty:
    print(f"  Long:  {(signals_df['direction'] == 'long').sum()}")
    print(f"  Short: {(signals_df['direction'] == 'short').sum()}")

## 6. Trade Execution

Execute each signal and produce the **standard trades DataFrame**
compatible with `shared.metrics`.

In [ ]:
def execute_fvg_trades(signals_df, market_data,
                       starting_capital=STARTING_CAPITAL,
                       include_fees=INCLUDE_FEES):
    """
    Execute FVG trades and return standardized trades DataFrame.

    Walks forward from entry bar, checks stop/target/EOD, sizes positions
    by risk, and tracks equity.
    """
    results = []
    equity = starting_capital

    for _, signal in signals_df.sort_values("entry_time").iterrows():
        date = signal["date"]
        entry_time  = signal["entry_time"]
        entry_price = signal["entry_price"]
        stop_loss   = signal["stop_loss"]
        target      = signal["profit_target"]
        risk        = signal["risk"]
        is_long     = signal["direction"] == "long"

        day_data = market_data[market_data["timestamp"].dt.date == date].reset_index(drop=True)
        entry_mask = day_data["timestamp"] == entry_time
        if entry_mask.sum() == 0:
            continue
        entry_idx = day_data[entry_mask].index[0]
        trade_data = day_data.iloc[entry_idx:]

        exit_time = exit_price = exit_reason = None
        for _, bar in trade_data.iterrows():
            if is_long:
                if bar["low"] <= stop_loss:
                    exit_time, exit_price, exit_reason = bar["timestamp"], stop_loss, "stop"
                    break
                if bar["high"] >= target:
                    exit_time, exit_price, exit_reason = bar["timestamp"], target, "target"
                    break
            else:
                if bar["high"] >= stop_loss:
                    exit_time, exit_price, exit_reason = bar["timestamp"], stop_loss, "stop"
                    break
                if bar["low"] <= target:
                    exit_time, exit_price, exit_reason = bar["timestamp"], target, "target"
                    break

        if exit_time is None:
            exit_time  = trade_data.iloc[-1]["timestamp"]
            exit_price = trade_data.iloc[-1]["close"]
            exit_reason = "eod"

        # Position sizing
        if risk <= 0 or equity <= 0:
            continue

        shares_by_risk = (equity * RISK_PERCENT) / risk
        shares_by_lev  = (equity * LEVERAGE) / entry_price
        shares = int(min(shares_by_risk, shares_by_lev))
        cap_used = (shares * entry_price) / LEVERAGE
        if cap_used > equity * MAX_CAP_PERCENT:
            shares = int(equity * MAX_CAP_PERCENT * LEVERAGE / entry_price)
        if shares <= 0:
            continue

        # P&L
        position = "long" if is_long else "short"
        if is_long:
            gross = shares * (exit_price - entry_price)
        else:
            gross = shares * (entry_price - exit_price)

        fees = calculate_fees(shares, entry_price, exit_price, position) if include_fees else 0.0
        net = gross - fees
        eq_before = equity
        equity += net

        results.append({
            "entry_time":   entry_time,
            "exit_time":    exit_time,
            "position":     position,
            "entry_price":  entry_price,
            "exit_price":   exit_price,
            "exit_reason":  exit_reason,
            "risk":         risk,
            "shares":       shares,
            "gross_pnl":    gross,
            "fees":         fees,
            "net_pnl":      net,
            "equity_before": eq_before,
            "equity":       equity,
            # Strategy-specific (optional)
            "direction":    signal["direction"],
            "fvg_range":    signal["fvg_range"],
        })

    return pd.DataFrame(results) if results else pd.DataFrame()


results_df = execute_fvg_trades(signals_df, qqq_data)
print(f"Executed {len(results_df)} trades")

## 7. Results

In [ ]:
# Combined metrics
metrics_all = evaluate_strategy(results_df, "FVG — All Trades")
print_metrics(metrics_all)

In [ ]:
# Long vs Short
long_df = results_df[results_df["position"] == "long"]
short_df = results_df[results_df["position"] == "short"]

metrics_list = [metrics_all]
if not long_df.empty:
    m_long = evaluate_strategy(long_df, "FVG — Long Only", starting_capital=STARTING_CAPITAL)
    metrics_list.append(m_long)
if not short_df.empty:
    m_short = evaluate_strategy(short_df, "FVG — Short Only", starting_capital=STARTING_CAPITAL)
    metrics_list.append(m_short)

print(compare_strategies(metrics_list))

## 8. Equity Curve & Returns

In [ ]:
plot_equity_curve(results_df, label="FVG — All Trades",
                  starting_capital=STARTING_CAPITAL)
plot_trade_returns(results_df, title="FVG — Per-Trade Returns")
plot_yearly_returns(metrics_all, title="FVG — Yearly Returns")

## 9. Exit Reason Breakdown

In [ ]:
for direction in ["long", "short"]:
    subset = results_df[results_df["position"] == direction]
    if subset.empty:
        continue
    print(f"\n{direction.upper()} TRADES — Exit Breakdown:")
    for reason in ["target", "stop", "eod"]:
        r = subset[subset["exit_reason"] == reason]
        if r.empty:
            continue
        avg_pnl = r["net_pnl"].mean()
        print(f"  {reason:<8} — {len(r):3d} trades ({len(r)/len(subset)*100:5.1f}%), "
              f"avg PnL: ${avg_pnl:+,.0f}")

## 10. Save Results

In [ ]:
save_trades(results_df, "fvg")

## 11. Conclusion

Modest edge, not compelling enough to pursue further. Documented for reference.
If revisited: try ATR-based stops and test on other instruments.